# BERT-Based Sentiment & Semantic Analysis

import

In [10]:
import pandas as pd
import numpy as np
import torch
import re
import os
import warnings
from transformers import pipeline, AutoTokenizer, AutoModel
from sklearn.metrics import classification_report
warnings.filterwarnings('ignore')

load data

In [11]:
DATA_PATH = "../data/processed/cleaned_reviews.csv"
MODEL_SAVE_PATH = "../models/bert_sentiment_model/"
OUTPUT_CSV = "../output/recommendations.csv"
EMBEDDINGS_PATH = "../output/embeddings/product_embeddings.npy"

os.makedirs("../models/bert_sentiment_model/", exist_ok=True)
os.makedirs("../output/embeddings/", exist_ok=True)
os.makedirs("../output/figures/", exist_ok=True)

Data Loading & Balanced Sampling

In [12]:
df = pd.read_csv(DATA_PATH, usecols=["productId", "text", "rating"])

def rating_to_sentiment(rating):
    if rating >= 4: return "positive"
    elif rating == 3: return "neutral"
    else: return "negative"

df["sentiment_label"] = df["rating"].map(rating_to_sentiment)

SAMPLE_SIZE = 1500 
PER_CLASS = SAMPLE_SIZE // 3
df_sample = df.groupby("sentiment_label").apply(lambda x: x.sample(min(len(x), PER_CLASS), random_state=42)).reset_index(drop=True)

print(f"Total rows loaded: {len(df):,}")
print(f"Sample size for CPU: {len(df_sample)}")

Total rows loaded: 326,447
Sample size for CPU: 1500


Text Cleaning

In [13]:
def clean_text(text):
    text = re.sub(r"<.*?>", " ", str(text))
    text = re.sub(r"\s+", " ", text).strip()
    return " ".join(text.split()[:300]) 

df_sample["text_clean"] = df_sample["text"].apply(clean_text)
print("Text cleaning complete.")

Text cleaning complete.


# Sentiment Analysis with BERT

In [14]:
MODEL_NAME = "nlptown/bert-base-multilingual-uncased-sentiment"
sentiment_pipeline = pipeline(
    task="text-classification",
    model=MODEL_NAME,
    truncation=True,
    max_length=512,
    device=-1 
)

def label_to_sentiment(bert_label):
    stars = int(bert_label[0])
    if stars >= 4: return "positive"
    elif stars == 3: return "neutral"
    else: return "negative"

print("Starting BERT Sentiment Inference...")
raw_results = sentiment_pipeline(df_sample["text_clean"].tolist(), batch_size=8)

df_sample["bert_sentiment"] = [label_to_sentiment(r["label"]) for r in raw_results]
df_sample["bert_score"] = [round(r["score"], 4) for r in raw_results]

print("\n Classification Report (BERT vs Original Rating) ")
print(classification_report(df_sample["sentiment_label"], df_sample["bert_sentiment"]))

Device set to use cpu


Starting BERT Sentiment Inference...

=== Classification Report (BERT vs Original Rating) ===
              precision    recall  f1-score   support

    negative       0.73      0.83      0.77       500
     neutral       0.75      0.53      0.62       500
    positive       0.78      0.89      0.83       500

    accuracy                           0.75      1500
   macro avg       0.75      0.75      0.74      1500
weighted avg       0.75      0.75      0.74      1500



# Semantic Understanding :Extracting Embeddings

In [15]:
print("Extracting Semantic Embeddings...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)

def get_embeddings(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128, padding=True)
    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.last_hidden_state[:, 0, :].numpy()

embeddings = np.vstack([get_embeddings(t) for t in df_sample["text_clean"].head(300)])

np.save(EMBEDDINGS_PATH, embeddings)
print(f"Embeddings saved to: {EMBEDDINGS_PATH}")

Extracting Semantic Embeddings...
Embeddings saved to: ../output/embeddings/product_embeddings.npy


# Final Aggregation and Saving

In [16]:
score_map = {"positive": 1, "neutral": 0, "negative": -1}
df_sample["sentiment_score"] = df_sample["bert_sentiment"].map(score_map)

product_final = df_sample.groupby("productId").agg(
    review_count=("bert_sentiment", "count"),
    avg_sentiment=("sentiment_score", "mean"),
    final_sentiment=("bert_sentiment", lambda x: x.mode()[0])
).reset_index()

save result

In [17]:
product_final.to_csv(OUTPUT_CSV, index=False)
sentiment_pipeline.model.save_pretrained(MODEL_SAVE_PATH)
sentiment_pipeline.tokenizer.save_pretrained(MODEL_SAVE_PATH)

('../models/bert_sentiment_model/tokenizer_config.json',
 '../models/bert_sentiment_model/special_tokens_map.json',
 '../models/bert_sentiment_model/vocab.txt',
 '../models/bert_sentiment_model/added_tokens.json',
 '../models/bert_sentiment_model/tokenizer.json')